<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 1. Aspectos generales

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql import Window
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.clustering import KMeans
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline
import pandas as pd
import builtins
import os

In [2]:
# Inicializar Spark Session
spark = SparkSession.builder \
    .appName("ModeloPrescriptivoVentas") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "50") \
    .config("spark.memory.fraction", "0.8") \
    .getOrCreate()

spark.sparkContext.setCheckpointDir("/tmp/spark_checkpoint")

In [3]:
# Rutas arquitectura medallion
BRONCE = "./bronce"
SILVER = "./silver"
GOLD   = "./gold"

import os
os.makedirs(SILVER, exist_ok=True)
os.makedirs(GOLD,   exist_ok=True)

In [4]:
df_articles = spark.read.csv(f"{BRONCE}/articles.csv"
						, sep = ","
						, header = True
						, inferSchema = True
						, nullValue = "NA"
						)

In [5]:
df_articles.head(3)

[Row(article_id=108775015, product_code=108775, prod_name='Strap top', product_type_no=253, product_type_name='Vest top', product_group_name='Garment Upper body', graphical_appearance_no=1010016, graphical_appearance_name='Solid', colour_group_code=9, colour_group_name='Black', perceived_colour_value_id=4, perceived_colour_value_name='Dark', perceived_colour_master_id=5, perceived_colour_master_name='Black', department_no=1676, department_name='Jersey Basic', index_code='A', index_name='Ladieswear', index_group_no=1, index_group_name='Ladieswear', section_no=16, section_name='Womens Everyday Basics', garment_group_no=1002, garment_group_name='Jersey Basic', detail_desc='Jersey top with narrow shoulder straps.'),
 Row(article_id=108775044, product_code=108775, prod_name='Strap top', product_type_no=253, product_type_name='Vest top', product_group_name='Garment Upper body', graphical_appearance_no=1010016, graphical_appearance_name='Solid', colour_group_code=10, colour_group_name='White'

In [6]:
schema = StructType([
	StructField("customer_id", StringType(), nullable=True),
	StructField("FN", DoubleType(), nullable=True),
	StructField("Active", DoubleType(), nullable=True),
	StructField("club_member_status", StringType(), nullable=True),
	StructField("fashion_news_frequency", StringType(), nullable=True),
	StructField("age", IntegerType(), nullable=True),
	StructField("postal_code", StringType(), nullable=True)
])

In [7]:
# Leer el CSV con el esquema personalizado
df_customers = spark.read.csv(f"{BRONCE}/customers.csv", header=True, schema=schema)

In [8]:
df_customers.head(3)

[Row(customer_id='00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657', FN=None, Active=None, club_member_status='ACTIVE', fashion_news_frequency='NONE', age=49, postal_code='52043ee2162cf5aa7ee79974281641c6f11a68d276429a91f8ca0d4b6efa8100'),
 Row(customer_id='0000423b00ade91418cceaf3b26c6af3dd342b51fd051eec9c12fb36984420fa', FN=None, Active=None, club_member_status='ACTIVE', fashion_news_frequency='NONE', age=25, postal_code='2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93f4c830291c32bc3057'),
 Row(customer_id='000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318', FN=None, Active=None, club_member_status='ACTIVE', fashion_news_frequency='NONE', age=24, postal_code='64f17e6a330a85798e4998f62d0930d14db8db1c054af6c9090f7dd3e38380dc')]

In [9]:
schema_transactions = StructType([
    StructField("t_dat", DateType(), nullable=True),
    StructField("customer_id", StringType(), nullable=True),
    StructField("article_id", StringType(), nullable=True),
    StructField("price", DoubleType(), nullable=True),
    StructField("sales_channel_id", StringType(), nullable=True)
])

In [10]:
df_transactions = spark.read.option("mergeSchema", "true") \
				.parquet(f"{BRONCE}/transactions.parquet")

In [11]:
df_transactions.show(2)

+----------+--------------------+----------+------------------+----------------+
|     t_dat|         customer_id|article_id|             price|sales_channel_id|
+----------+--------------------+----------+------------------+----------------+
|2018-09-20|000058a12d5b43e67...| 663713001|0.0508305084745762|               2|
|2018-09-20|000058a12d5b43e67...| 541518023|0.0304915254237288|               2|
+----------+--------------------+----------+------------------+----------------+
only showing top 2 rows



In [12]:
df_transactions.printSchema()

root
 |-- t_dat: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- article_id: long (nullable = true)
 |-- price: double (nullable = true)
 |-- sales_channel_id: long (nullable = true)



In [13]:
# Convertir la fecha al tipo correcto y multiplicar price por 50000
df_transactions = df_transactions \
    .withColumn("t_dat", to_date(col("t_dat"), "yyyy-MM-dd")) \
    .withColumn("price", col("price") * 50000)

In [14]:
# Mostrar valores únicos de CLUB_MEMBER_STATUS
df_customers.select(col("CLUB_MEMBER_STATUS")).distinct().show(truncate=False)

+------------------+
|CLUB_MEMBER_STATUS|
+------------------+
|PRE-CREATE        |
|LEFT CLUB         |
|ACTIVE            |
|null              |
+------------------+



In [15]:
# reemplazar 'lEFT CLUB' por 'PREMIUM' en CLUB_MEMBER_STATUS
df_customers = df_customers.withColumn(
    "CLUB_MEMBER_STATUS",
    expr("CASE WHEN CLUB_MEMBER_STATUS = 'LEFT CLUB' THEN 'PREMIUM' ELSE CLUB_MEMBER_STATUS END")
)

In [16]:
# Ajustar los datos de las columnas
# CPrimero convertir a string si es necesario, luego reemplazar
df_customers = df_customers.withColumn(
    "Active",
    when(col("Active").cast("string") == "1", "ACTIVE")
    .when(col("Active").cast("string") == "0", "INACTIVE")
    .otherwise(col("Active"))
)


In [17]:
# AGREGAR COSTO - COMO SUPUESTO DEL PRECIO MAXIMO DE LAS VENTAS 80 AL 90%
# Usar semilla para que los números aleatorios sean reproducibles
price_article = (
    df_transactions.groupBy("article_id")
    .agg(max("price").alias("max_price"))
    .withColumn("factor_aleatorio", rand(42) * 0.1 + 0.8)
    .withColumn("cost", col("max_price") * col("factor_aleatorio"))
)

In [18]:
df_articles.head(3)

[Row(article_id=108775015, product_code=108775, prod_name='Strap top', product_type_no=253, product_type_name='Vest top', product_group_name='Garment Upper body', graphical_appearance_no=1010016, graphical_appearance_name='Solid', colour_group_code=9, colour_group_name='Black', perceived_colour_value_id=4, perceived_colour_value_name='Dark', perceived_colour_master_id=5, perceived_colour_master_name='Black', department_no=1676, department_name='Jersey Basic', index_code='A', index_name='Ladieswear', index_group_no=1, index_group_name='Ladieswear', section_no=16, section_name='Womens Everyday Basics', garment_group_no=1002, garment_group_name='Jersey Basic', detail_desc='Jersey top with narrow shoulder straps.'),
 Row(article_id=108775044, product_code=108775, prod_name='Strap top', product_type_no=253, product_type_name='Vest top', product_group_name='Garment Upper body', graphical_appearance_no=1010016, graphical_appearance_name='Solid', colour_group_code=10, colour_group_name='White'

In [19]:
df_articles  = df_articles.alias("a").join(
    price_article.select("article_id", "cost").alias("p"),
    col("a.article_id") == col("p.article_id"),
    how="left"
).select(
    "a.*",  # Todas las columnas de df_articles
    "p.cost"  # Solo la columna cost de price_article
)


In [20]:
# --- CAPA SILVER: guardar datos limpios y enriquecidos ---
df_transactions.write.mode("overwrite").parquet(f"{SILVER}/transactions")
df_customers.write.mode("overwrite").parquet(f"{SILVER}/customers")
df_articles.write.mode("overwrite").parquet(f"{SILVER}/articles")

# Leer desde silver para el resto del pipeline
df_transactions = spark.read.parquet(f"{SILVER}/transactions")
df_customers    = spark.read.parquet(f"{SILVER}/customers")
df_articles     = spark.read.parquet(f"{SILVER}/articles")

<a id='2'></a>
<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 2. Feature Engineering en PySpark

<div style="background-color:#9B9B9B; color:white; padding:10px; border-radius:5px; font-size:13px">

## 1. Contexto General
---

La clase **FeatureEngineeringSpark** está diseñada para automatizar el proceso de generación de variables (*features*) a partir de tres fuentes principales de datos:

- **df_articles**: información de artículos o productos vendidos.
- **df_customers**: información demográfica y de membresía de los clientes.
- **df_ventas**: transacciones históricas (ventas realizadas por cliente y artículo).

El objetivo del modelo es preparar un DataFrame listo para modelar (por ejemplo, para un modelo de propensión de compra), aplicando transformaciones, agregaciones y normalizaciones de manera distribuida con PySpark.

## 2. Estructura General de la Clase
---

.. code-block:: python

    class FeatureEngineeringSpark:
        ├── __init__                  # Inicializa los datasets
        ├── crear_features_clientes   # Genera variables derivadas de comportamiento
        ├── crear_target_propension   # Define el target (quién compró recientemente)
        ├── preparar_features_ml      # Prepara y escala las features numéricas
        ├── get_feature_importance    # Extrae importancia de variables de un modelo
        └── verificar_datos_features  # Función auxiliar para revisar calidad de datos

Cada componente se explica a continuación.

## 3. Constructor: **__init__**
---

**Propósito:**

Inicializar las fuentes de datos y declarar el DataFrame de salida (**features_df**).

    def __init__(self, df_articles, df_customers, df_ventas):
        self.df_articles = df_articles
        self.df_customers = df_customers
        self.df_ventas = df_ventas
        self.features_df = None

**Función matemática:**

No aplica aún; es simplemente la inicialización del entorno y almacenamiento de referencias de los DataFrames.

## 4. Método **crear_features_clientes()**
---

**Objetivo:**

Construir variables de comportamiento y demográficas a nivel de cliente.

A. Variables agregadas por cliente
----------------------------------

El método agrupa las ventas por **customer_id** y calcula varias funciones de agregación:

- **Total de compras**:  
$\text{total\_compras}_i = \text{count}(\text{article\_id})$

- **Precio promedio**:  
$\text{precio\_promedio}_i = \dfrac{\sum \text{price}}{\text{count}(\text{price})}$

- **Gasto total**:  
$\text{gasto\_total}_i = \sum \text{price}$

- **Artículos únicos**:  
$\text{unique\_articles}_i = \text{countDistinct}(\text{article\_id})$

- **Última compra**:  
$\text{ultima\_compra}_i = \max(t\_dat)$


B. Frecuencia de compra (Recency)
---------------------------------

Se calcula la cantidad de días desde la última compra del cliente:

$\text{dias\_desde\_ultima\_compra}_i = \text{fecha\_max\_ventas} - \text{ultima\_compra}_i$

Esta variable es análoga al componente *Recency* del modelo RFM (*Recency, Frequency, Monetary*).

C. Variables demográficas
-------------------------

Transforma variables categóricas en numéricas binarias (0/1):

- **Es premium**:  
  $\text{is\_premium}_i = \begin{cases} 
      1 & \text{si } club\_member\_status = \text{"PREMIUM"} \\
      0 & \text{caso contrario}
   \end{cases}$

- **Está activo**:  
  $\text{is\_active}_i = \begin{cases} 
      1 & \text{si } Active = \text{"ACTIVE"} \\
      0 & \text{caso contrario}
   \end{cases}$

- **Recibe noticias regularmente**:  
  $\text{news\_regular}_i = \begin{cases} 
      1 & \text{si } fashion\_news\_frequency = \text{"Regularly"} \\
      0 & \text{caso contrario}
   \end{cases}$

D. Unión de fuentes
-------------------

Se unen los datos demográficos (**df_customers**) con los datos transaccionales agregados (**compras_cliente**) por **customer_id**.

Todos los valores nulos se reemplazan por valores neutros (por ejemplo, gasto = 0, recency = 365 días).


## 5. Método **crear_target_propension()**
---

**Objetivo:**

Construir la variable objetivo (*target*) que indica si un cliente compró en los últimos N días (por defecto, 30).

**Lógica temporal:**

- Fecha más reciente del dataset:  
  $\text{fecha\_max} = \max(t\_dat)$

- Horizonte de predicción:  
  $\text{fecha\_corte} = \text{fecha\_max} - \text{horizonte\_días}$

- Target binario:  
  $\text{compro\_reciente}_i = \begin{cases} 
      1 & \text{si el cliente } i \text{ tiene alguna venta posterior a } \text{fecha\_corte} \\
      0 & \text{en caso contrario}
   \end{cases}$


## 6. Método **preparar_features_ml()**
---

**Objetivo:**

Preparar las variables numéricas para el modelado mediante:

- Limpieza de nulos
- Vectorización
- Escalado estandarizado

### A. Selección de columnas numéricas
----------------------------------

$X = [\text{age}, \text{FN}, \text{is\_premium}, \text{is\_active}, \text{news\_regular}, \text{total\_compras}, \text{precio\_promedio}, \text{gasto\_total}, \text{unique\_articles}, \text{dias\_desde\_ultima\_compra}]$

### B. Ensamblado vectorial
-----------------------

Se utiliza el transformador **VectorAssembler** de Spark:

$\text{features\_raw}_i = [x_{i1}, x_{i2}, ..., x_{i10}]$

Esto convierte las 10 columnas numéricas en un único vector denso por fila.

### C. Escalado (Estandarización)
-----------------------------

Se aplica **StandardScaler**, que transforma cada variable según la fórmula:

$x' = \dfrac{x - \mu}{\sigma}$

Donde:
- $\mu$: media de la columna  
- $\sigma$: desviación estándar

Este proceso garantiza que todas las variables tengan media 0 y desviación 1, eliminando efectos de magnitud desigual.

## 7. Método **get_feature_importance(model)**
---

**Objetivo:**

Extraer y ordenar la importancia relativa de cada *feature* a partir de modelos que lo permitan (por ejemplo, **RandomForestClassifier**).

**Cálculo:**

Si el modelo posee el atributo **.featureImportances**, Spark devuelve un vector  
$I = [i_1, i_2, ..., i_{10}]$ donde cada elemento representa:

$i_j = \dfrac{\text{ganancia de información aportada por la variable } j}{\text{suma total de ganancias}}$

## 8. Función auxiliar **verificar_datos_features()**
---

**Objetivo:**

Evaluar la calidad de los datos antes del modelado.
- Conteo de registros
- Conteo de nulos por columna
- Estadísticas descriptivas (media, desviación, mínimo, máximo)
- Distribución del *target*

**Matemáticamente**, se ejecutan operaciones de:

$\text{count}(x),\ \text{mean}(x),\ \text{stddev}(x),\ \text{min}(x),\ \text{max}(x)$


In [21]:
class FeatureEngineeringSpark:
    """
    Clase para feature engineering escalable en PySpark
    """
    
    def __init__(self, df_articles, df_customers, df_ventas):
        self.df_articles = df_articles
        self.df_customers = df_customers
        self.df_ventas = df_ventas
        self.features_df = None
        
    def crear_features_clientes(self):
        """
        Crea features de comportamiento de clientes
        """
        print("Creando features de clientes...")
        
        # Historial de compras por cliente
        compras_cliente = self.df_ventas.groupBy("customer_id").agg(
            count("article_id").alias("total_compras"),
            mean("price").alias("precio_promedio"),
            sum("price").alias("gasto_total"),
            countDistinct("article_id").alias("unique_articles"),
            max("t_dat").alias("ultima_compra")
        )
        
        # Frecuencia de compras (días desde última compra)
        fecha_max = self.df_ventas.agg(max("t_dat")).collect()[0][0]
        compras_cliente = compras_cliente.withColumn(
            "dias_desde_ultima_compra", 
            datediff(lit(fecha_max), "ultima_compra")
        )
        
        # Features demográficas y de segmentación - ASEGURAR TIPOS NUMÉRICOS
        clientes_features = self.df_customers \
            .withColumn("is_premium", 
                       when(col("club_member_status") == "PREMIUM", 1).otherwise(0).cast("double")) \
            .withColumn("is_active", 
                       when(col("Active") == "ACTIVE", 1).otherwise(0).cast("double")) \
            .withColumn("news_regular", 
                       when(col("fashion_news_frequency") == "Regularly", 1).otherwise(0).cast("double")) \
            .withColumn("age", col("age").cast("double")) \
            .withColumn("FN", col("FN").cast("double"))
        
        # Unir todas las features
        self.features_df = clientes_features.join(compras_cliente, "customer_id", "left")
        
        # Llenar nulos para clientes sin compras
        self.features_df = self.features_df.fillna({
            'total_compras': 0.0,
            'precio_promedio': 0.0,
            'gasto_total': 0.0,
            'unique_articles': 0.0,
            'dias_desde_ultima_compra': 365.0
        })
        
        # Asegurar que todas las columnas sean del tipo correcto
        self.features_df = self.features_df \
            .withColumn("total_compras", col("total_compras").cast("double")) \
            .withColumn("precio_promedio", col("precio_promedio").cast("double")) \
            .withColumn("gasto_total", col("gasto_total").cast("double")) \
            .withColumn("unique_articles", col("unique_articles").cast("double")) \
            .withColumn("dias_desde_ultima_compra", col("dias_desde_ultima_compra").cast("double"))
        
        print("Features de clientes creados correctamente")
        print("Esquema de features:")
        self.features_df.printSchema()
        
        return self.features_df
    
    def crear_target_propension(self, horizonte_dias=30):
        """
        Crea target para modelo de propensión (compras recientes)
        """
        print("Creando target de propensión...")
        
        # Fecha de corte para target
        fecha_max = self.df_ventas.agg(max("t_dat")).collect()[0][0]
        fecha_corte = fecha_max - pd.Timedelta(days=horizonte_dias)
        
        # Clientes que compraron en el período reciente
        compras_recientes = self.df_ventas.filter(col("t_dat") > lit(fecha_corte)) \
            .select("customer_id").distinct() \
            .withColumn("compro_reciente", lit(1).cast("double"))
        
        # Unir con features
        self.features_df = self.features_df.join(
            compras_recientes, "customer_id", "left"
        ).fillna({'compro_reciente': 0.0})
        
        print(f"Target creado: {self.features_df.filter(col('compro_reciente') == 1).count():,} clientes compraron recientemente")
        
        return self.features_df
    
    def preparar_features_ml(self):
        """
        Prepara features para modelos de ML
        """
        print("Preparando features para ML...")

        feature_columns = [
            'age', 'FN', 'is_premium', 'is_active', 'news_regular',
            'total_compras', 'precio_promedio', 'gasto_total',
            'unique_articles', 'dias_desde_ultima_compra'
        ]

        # Limpiar nulos y hacer checkpoint para cortar el linaje
        features_clean = self.features_df.dropna(subset=feature_columns)
        features_clean = features_clean.repartition(50).checkpoint()

        # Ensamblar vector de features
        assembler = VectorAssembler(
            inputCols=feature_columns,
            outputCol="features_raw",
            handleInvalid="skip"
        )

        # withMean=True requiere materializar todo en driver; usar withMean=False
        scaler = StandardScaler(
            inputCol="features_raw",
            outputCol="features",
            withStd=True,
            withMean=False
        )

        pipeline = Pipeline(stages=[assembler, scaler])
        self.features_model = pipeline.fit(features_clean)
        self.features_df = self.features_model.transform(features_clean)

        # Cachear resultado para evitar recomputación en pasos siguientes
        self.features_df.cache()
        print("Features para ML preparados correctamente")

        return self.features_df
    
    def get_feature_importance(self, model):
        """
        Obtener importancia de features (para modelos que lo soporten)
        """
        try:
            if hasattr(model, 'featureImportances'):
                feature_importances = model.featureImportances
                feature_columns = [
                    'age', 'FN', 'is_premium', 'is_active', 'news_regular',
                    'total_compras', 'precio_promedio', 'gasto_total', 
                    'unique_articles', 'dias_desde_ultima_compra'
                ]
                
                # Crear DataFrame con importancia de features
                importance_list = [(col, float(importance)) 
                                 for col, importance in zip(feature_columns, feature_importances)]
                
                importance_df = spark.createDataFrame(
                    importance_list, 
                    ["feature", "importance"]
                ).orderBy(col("importance").desc())
                
                return importance_df
            else:
                print("El modelo no tiene atributo featureImportances")
                return None
        except Exception as e:
            print(f"Error al obtener importancia de features: {e}")
            return None

# Función auxiliar para verificar datos
def verificar_datos_features(features_df):
    """
    Verifica la calidad de los datos en el DataFrame de features
    """
    print("=== VERIFICACIÓN DE DATOS ===")
    
    # Verificar conteo
    print(f"Total de registros: {features_df.count():,}")
    
    # Verificar nulos
    print("\nValores nulos por columna:")
    feature_columns = [
        'age', 'FN', 'is_premium', 'is_active', 'news_regular',
        'total_compras', 'precio_promedio', 'gasto_total', 
        'unique_articles', 'dias_desde_ultima_compra', 'compro_reciente'
    ]
    
    for col_name in feature_columns:
        if col_name in features_df.columns:
            nulos = features_df.filter(col(col_name).isNull()).count()
            print(f"  {col_name}: {nulos} nulos")
    
    # Verificar estadísticas básicas
    print("\nEstadísticas descriptivas:")
    features_df.select(feature_columns).describe().show()
    
    # Verificar distribución del target
    if 'compro_reciente' in features_df.columns:
        print("\nDistribución del target:")
        features_df.groupBy("compro_reciente").count().orderBy("compro_reciente").show()

In [22]:
# Ejecutar feature engineering con verificación
try:
    feature_engineer = FeatureEngineeringSpark(df_articles, df_customers, df_transactions)
    features_df = feature_engineer.crear_features_clientes()
    features_df = feature_engineer.crear_target_propension()
    features_df = feature_engineer.preparar_features_ml()
    
except Exception as e:
    print(f"Error durante el feature engineering: {e}")
    import traceback
    traceback.print_exc()


Creando features de clientes...
Features de clientes creados correctamente
Esquema de features:
root
 |-- customer_id: string (nullable = true)
 |-- FN: double (nullable = true)
 |-- Active: string (nullable = true)
 |-- CLUB_MEMBER_STATUS: string (nullable = true)
 |-- fashion_news_frequency: string (nullable = true)
 |-- age: double (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- is_premium: double (nullable = false)
 |-- is_active: double (nullable = false)
 |-- news_regular: double (nullable = false)
 |-- total_compras: double (nullable = true)
 |-- precio_promedio: double (nullable = false)
 |-- gasto_total: double (nullable = false)
 |-- unique_articles: double (nullable = true)
 |-- ultima_compra: date (nullable = true)
 |-- dias_desde_ultima_compra: double (nullable = true)

Creando target de propensión...
Target creado: 245,945 clientes compraron recientemente
Preparando features para ML...
Features para ML preparados correctamente


In [23]:
# Mostrar resultado
print("Features creadas:")
features_df.select("customer_id"
                   , "age"
                   , "total_compras"
                   , "gasto_total"
                   , "features").head(5)


Features creadas:


[Row(customer_id='154a8a417a9e475789619cf19302bb50b9ca97b22feb4b568a930b0de64e29af', age=26.0, total_compras=6.0, gasto_total=6791.525423728799, features=DenseVector([1.7465, 0.0, 0.0, 0.0, 17.2132, 0.1316, 1.7007, 0.0976, 0.1627, 0.0])),
 Row(customer_id='6ac97351d133dda051fe8283104f7453876a29e972ff4223b11154eba35e8d92', age=39.0, total_compras=8.0, gasto_total=11603.389830508446, features=DenseVector([2.6197, 0.0, 0.0, 0.0, 17.2132, 0.1755, 2.1792, 0.1667, 0.1899, 3.1611])),
 Row(customer_id='12cf7d69bfa261e356ab0c40c457ba326e862ced0a99a1434cf58b7d2302699c', age=31.0, total_compras=48.0, gasto_total=75298.30508474565, features=DenseVector([2.0823, 0.0, 0.0, 0.0, 17.2132, 1.0529, 2.3569, 1.0818, 1.1663, 0.1634])),
 Row(customer_id='484a772cf1dda659623ff2bb2812e69c617f27f89b67785eabc9e668e79b30e6', age=62.0, total_compras=2.0, gasto_total=1218.644067796605, features=DenseVector([4.1647, 0.0, 0.0, 0.0, 17.2132, 0.0439, 0.9155, 0.0175, 0.0542, 0.0])),
 Row(customer_id='602f5148997774fa5a

<a id='3'></a>
<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 3. Modelo de Propensión de Compra en PySpark

<div style="background-color:#9B9B9B; color:white; padding:10px; border-radius:5px; font-size:13px">

## 1. Contexto General
---

La clase **ModeloPropensionSpark** implementa un modelo de **propensión de compra** utilizando las herramientas de machine learning distribuido de PySpark (MLlib).  
El objetivo es predecir, para cada cliente, la **probabilidad de haber realizado una compra reciente**, basándose en un conjunto de características (*features*) previamente generadas (por ejemplo, mediante la clase **FeatureEngineeringSpark**).

Este modelo emplea un **Random Forest Regressor**, adecuado para capturar relaciones no lineales y manejar conjuntos de datos con alta dimensionalidad o ruido.

## 2. Estructura General de la Clase
---

    class ModeloPropensionSpark:
        ├── __init__                        # Inicializa contenedores del modelo y pipeline
        ├── entrenar_modelo_propension      # Entrena y evalúa el modelo en Spark
        └── predecir_propension_cliente     # Consulta la probabilidad para un cliente específico

## 3. Constructor: **__init__**
---

**Propósito:**

Inicializar los atributos internos que almacenarán el modelo entrenado y, opcionalmente, el pipeline de transformación.

    def __init__(self):
        self.model = None
        self.pipeline = None

Al inicio, tanto el modelo como el pipeline están vacíos (**None**), indicando que aún no se ha realizado ningún entrenamiento.

## 4. Método **entrenar_modelo_propension(features_df, test_size=0.2)**
---

**Objetivo:**

Entrenar un modelo de regresión basado en **Random Forest** para predecir la variable objetivo **compro_reciente** (codificada como 0/1) a partir del vector de características **features**.

### A. División de datos
--------------------

Se divide aleatoriamente el DataFrame de entrada en conjuntos de entrenamiento y prueba:

- **Entrenamiento**: $(1 - \text{test\_size}) \times 100\%$ de los datos  
- **Prueba**: $\text{test\_size} \times 100\%$ de los datos  

Se fija una semilla aleatoria (**seed=42**) para garantizar reproducibilidad.

### B. Configuración del modelo
---------------------------

Se instancia un **RandomForestRegressor** con los siguientes hiperparámetros:

- **featuresCol="features"**: columna que contiene el vector de características.
- **labelCol="compro_reciente"**: variable objetivo binaria (tratada como continua en regresión).
- **numTrees=100**: número de árboles en el bosque.
- **maxDepth=10**: profundidad máxima de cada árbol.
- **seed=42**: semilla para inicialización aleatoria.

> Nota: Aunque el target es binario, se usa un **regresor** porque la salida deseada es una *probabilidad estimada* (valor continuo entre 0 y 1), no una clasificación dura.

### C. Entrenamiento y evaluación
-----------------------------

1. El modelo se entrena con el conjunto de entrenamiento.
2. Se generan predicciones sobre el conjunto de prueba.
3. Se calcula el **RMSE** (*Root Mean Squared Error*) usando **RegressionEvaluator**:

$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$

   Donde:
   - $y_i$: valor real de **compro_reciente**
   - $\hat{y}_i$: predicción del modelo

### D. Predicción sobre todo el dataset
-----------------------------------

Una vez entrenado, el modelo transforma el **DataFrame completo** (**features_df**), añadiendo una nueva columna con las predicciones. Esta columna se renombra a **probabilidad_compra** para claridad semántica.


## 5. Método **predecir_propension_cliente(customer_id)**
---

**Objetivo:**

Consultar la probabilidad de compra reciente para un cliente específico, identificado por su **customer_id**.

**Lógica:**

1. Verifica que el modelo haya sido entrenado (**self.features_df** no debe ser **None**).
2. Filtra el DataFrame por el **customer_id** solicitado.
3. Extrae el valor de **probabilidad_compra**.

**Comportamiento ante ausencia de cliente:**

- Si el cliente **no existe** en el DataFrame, devuelve **0.0** como valor por defecto (interpretado como "muy baja propensión").


## 6. Consideraciones Técnicas y Limitaciones
---

- **Regresión vs Clasificación**: Aunque el problema es inherentemente de clasificación binaria, se opta por regresión para obtener una estimación continua de probabilidad. Para una solución más rigurosa, podría usarse **RandomForestClassifier** + método **.transform()** con columna **probability**, seguido de extracción del segundo componente del vector de probabilidades.
- **Escalabilidad**: Todo el proceso se ejecuta en modo distribuido gracias a PySpark, permitiendo manejar millones de clientes sin problemas de memoria.
- **Persistencia**: La clase actual **no incluye métodos para guardar/cargar** el modelo. En producción, se recomienda usar **model.write().save(path)** y **RandomForestRegressionModel.load(path)**.

**Salida esperada tras entrenamiento:**

Un modelo listo para generar scores de propensión a escala, con capacidad de consulta individual y métricas de desempeño básicas (RMSE).

In [24]:
class ModeloPropensionSpark:
    """
    Modelo de propensión de compra usando MLlib de PySpark
    """
    
    def __init__(self):
        self.model = None
        self.pipeline = None
        
    def entrenar_modelo_propension(self, features_df, test_size=0.2):
        """
        Entrena modelo Random Forest para propensión de compra
        """
        print("Entrenando modelo de propensión en PySpark...")
        
        # Dividir datos en entrenamiento y prueba
        train_df, test_df = features_df.randomSplit([1-test_size, test_size], seed=42)
        
        # Crear pipeline con Random Forest
        rf = RandomForestRegressor(
            featuresCol="features",
            labelCol="compro_reciente",
            numTrees=100,
            maxDepth=10,
            seed=42
        )
        
        # Entrenar modelo
        self.model = rf.fit(train_df)
        
        # Evaluar modelo
        predictions = self.model.transform(test_df)
        
        # Calcular métricas simples
        from pyspark.ml.evaluation import RegressionEvaluator
        evaluator = RegressionEvaluator(
            labelCol="compro_reciente",
            predictionCol="prediction",
            metricName="rmse"
        )
        
        rmse = evaluator.evaluate(predictions)
        print(f"RMSE del modelo: {rmse:.4f}")
        
        # Añadir predicciones al DataFrame completo
        self.features_df = self.model.transform(features_df)
        self.features_df = self.features_df.withColumnRenamed("prediction", "probabilidad_compra")
        
        print("Modelo de propensión entrenado")
        return self.features_df
    
    def predecir_propension_cliente(self, customer_id):
        """
        Predice propensión para un cliente específico
        """
        if self.features_df is None:
            raise ValueError("Modelo no entrenado")
            
        resultado = self.features_df.filter(col("customer_id") == customer_id) \
            .select("customer_id", "probabilidad_compra") \
            .collect()
            
        if resultado:
            return resultado[0]["probabilidad_compra"]
        else:
            return 0.0


In [25]:
# Entrenar modelo de propensión
modelo_propension = ModeloPropensionSpark()
features_df = modelo_propension.entrenar_modelo_propension(features_df)


Entrenando modelo de propensión en PySpark...
RMSE del modelo: 0.0043
Modelo de propensión entrenado


In [26]:
# Mostrar algunas predicciones
print("\nPredicciones de ejemplo:")
features_df.select("customer_id", "compro_reciente", "probabilidad_compra") \
    .orderBy(desc("probabilidad_compra")) \
    .show(10)



Predicciones de ejemplo:
+--------------------+---------------+-------------------+
|         customer_id|compro_reciente|probabilidad_compra|
+--------------------+---------------+-------------------+
|8673da1ac0db56acf...|            1.0|                1.0|
|4f84b158643b4d59f...|            1.0|                1.0|
|018a18f7b0755d8d3...|            1.0|                1.0|
|42f086703ad82f65c...|            1.0|                1.0|
|4f438fa71a1de7bbe...|            1.0|                1.0|
|876721fc8e5da4e3b...|            1.0|                1.0|
|5169bbfb3097d2695...|            1.0|                1.0|
|efa466a1401d7eae5...|            1.0|                1.0|
|ed2131f7f0d8a0d10...|            1.0|                1.0|
|4a0badcc2e93e1749...|            1.0|                1.0|
+--------------------+---------------+-------------------+
only showing top 10 rows



<a id='4'></a>
<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 4. Segmentación de Clientes con K-Means en PySpark

<div style="background-color:#9B9B9B; color:white; padding:10px; border-radius:5px; font-size:13px">

## 1. Contexto General
======================

La clase **SegmentadorClientesSpark** implementa una solución de segmentación no supervisada de clientes utilizando el algoritmo **K-Means** en PySpark.  
El objetivo es agrupar a los clientes en segmentos homogéneos basados en sus características comportamentales y demográficas (previamente transformadas en un vector de features).  
Esta segmentación permite personalizar estrategias de marketing, retención o reactivación según el perfil de cada grupo.

## 2. Estructura General de la Clase
===================================

.. code-block:: python

    class SegmentadorClientesSpark:
        ├── __init__                # Configura número de segmentos
        ├── segmentar_clientes      # Aplica K-Means y asigna segmentos
        └── _caracterizar_segmentos # Analiza y nombra los segmentos encontrados

## 3. Constructor: __init__
===============================

**Propósito:**

Inicializar los hiperparámetros del modelo de clustering y variables internas.

.. code-block:: python

    def __init__(self, n_segmentos=4):
        self.n_segmentos = n_segmentos
        self.kmeans = None
        self.features_df = None

- **n_segmentos**: número de clusters a generar (por defecto: 4).
- **kmeans_model**: se inicializa durante el entrenamiento.
- **features_df**: almacena el DataFrame con las predicciones de segmento tras la ejecución.

## 4. Método segmentar_clientes(features_df)
------------------------------------------------

**Objetivo:**

Aplicar el algoritmo **K-Means** sobre un conjunto de clientes representado por un vector de características (**features**) y asignar a cada cliente un identificador de segmento.

### Fundamentos matemáticos de K-Means

El algoritmo **K-Means** es un método de *clustering particional no supervisado* que busca particionar $n$ observaciones en $k$ clusters disjuntos, de modo que cada observación pertenezca al cluster cuyo **centroide** (media del cluster) esté más cercano en términos de distancia euclidiana.

Dado un conjunto de puntos $\mathbf{X} = \{ \mathbf{x}_1, \mathbf{x}_2, \dots, \mathbf{x}_n \}$, donde cada $\mathbf{x}_i \in \mathbb{R}^d$ es un vector de $d$ características (por ejemplo: edad, gasto total, recencia, etc.), el objetivo de K-Means es encontrar un conjunto de centroides $\mathbf{C} = \{ \boldsymbol{\mu}_1, \boldsymbol{\mu}_2, \dots, \boldsymbol{\mu}_k \}$ que minimice la **función de inercia** (suma de distancias al cuadrado dentro de los clusters):


$\text{Inercia} = \sum_{j=1}^{k} \sum_{\mathbf{x}_i \in S_j} \| \mathbf{x}_i - \boldsymbol{\mu}_j \|^2$

donde:

- $S_j$ es el conjunto de observaciones asignadas al cluster $j$,
- $\boldsymbol{\mu}_j = \frac{1}{|S_j|} \sum_{\mathbf{x}_i \in S_j} \mathbf{x}_i$ es el **centroide** del cluster $j$,
- $\| \cdot \|$ denota la norma euclidiana.

El algoritmo itera los siguientes pasos hasta convergencia:

1. **Inicialización**: seleccionar $k$ centroides iniciales (en PySpark, por defecto se usa *k-means||*, una variante escalable de *k-means++*).
2. **Asignación**: asignar cada punto al cluster cuyo centroide esté más cercano.
3. **Actualización**: recalcular cada centroide como la media de los puntos asignados a su cluster.

En este contexto, cada cliente es un punto en un espacio multidimensional definido por sus *features* estandarizadas (por ejemplo: **gasto_total**, **dias_desde_ultima_compra**, **total_compras**, etc.), y el algoritmo agrupa a clientes con comportamientos similares.

### Pasos principales en la implementación

1. **Entrenamiento del modelo**:

   Se instancia y entrena un modelo **KMeans** de PySpark con los siguientes parámetros:

   - **featuresCol="features"**: columna que contiene el vector de características (previamente ensamblado y escalado).
   - **predictionCol="segmento"**: columna donde se almacenará la etiqueta del cluster asignado.
   - **k = self.n_segmentos**: número de clusters definido en la inicialización (por defecto: 4).
   - **seed = 42**: fija la semilla aleatoria para garantizar **reproducibilidad**.

2. **Transformación del DataFrame**:

   Se aplica el modelo entrenado al DataFrame de entrada para predecir el segmento de cada cliente, añadiendo la columna **segmento**.

3. **Caracterización de segmentos**:

   Se llama internamente a **_caracterizar_segmentos()** para analizar estadísticamente cada cluster y asignar un nombre semántico basado en su perfil promedio.

**Resultado:**

Devuelve un DataFrame enriquecido con dos nuevas columnas:

- **segmento**: identificador numérico del cluster (0 a $k-1$).
- **segmento_nombre**: nombre semántico asignado según el análisis de comportamiento (por ejemplo: **"CLIENTES_LEALES"**, **"CLIENTES_INACTIVOS"**, etc.).

Este enfoque permite pasar de una agrupación puramente matemática a una **segmentación comercial interpretable**, lista para accionar estrategias de marketing personalizadas.

## 5. Método privado _caracterizar_segmentos()
---

**Objetivo:**

Analizar estadísticamente cada segmento para comprender su perfil y asignar un nombre descriptivo.

**Agregaciones realizadas por segmento:**

- **num_clientes**: cantidad de clientes en el segmento.
- **edad_promedio**: media de edad.
- **gasto_promedio**: media del gasto total.
- **compras_promedio**: media del número total de compras.
- **propension_promedio**: media de la probabilidad de compra reciente.
- **inactividad_promedio**: media de días desde la última compra.

Estas métricas se obtienen mediante operaciones de agregación de Spark:

$text{mean}(x),\ \text{count}(x)$

**Asignación de nombres semánticos:**

Basado en el análisis típico de perfiles, se asignan nombres fijos a los primeros 4 segmentos:

| **segmento** | **segmento_nombre**        |
|-----------|------------------------|
| 0         | CLIENTES_LEALES        |
| 1         | CLIENTES_NUEVOS        |
| 2         | CLIENTES_INACTIVOS     |
| 3         | CLIENTES_POTENCIALES   |


> **Nota:** Esta asignación asume que los clusters se generan de forma consistente (gracias al **seed=42**). En escenarios dinámicos, se recomendaría una lógica adaptativa para nombrar segmentos.

**Salida visual:**

El método imprime en consola un resumen tabular de las características promedio por segmento usando **.show()**.

**Resultado final:**

El DataFrame interno **self.features_df** se actualiza con la columna **segmento_nombre**, permitiendo análisis posteriores o exportación para campañas segmentadas.


In [27]:
class SegmentadorClientesSpark:
    """
    Segmentación de clientes usando K-Means en PySpark
    """
    
    def __init__(self, n_segmentos=4):
        self.n_segmentos = n_segmentos
        self.kmeans = None
        self.features_df = None
        
    def segmentar_clientes(self, features_df):
        """
        Aplica clustering K-Means para segmentar clientes
        """
        print("Segmentando clientes con K-Means...")
        
        # Entrenar modelo K-Means
        kmeans = KMeans(
            featuresCol="features",
            predictionCol="segmento",
            k=self.n_segmentos,
            seed=42
        )
        
        self.kmeans_model = kmeans.fit(features_df)
        self.features_df = self.kmeans_model.transform(features_df)
        
        # Caracterizar segmentos
        self._caracterizar_segmentos()
        
        print("Segmentación completada")
        return self.features_df
    
    def _caracterizar_segmentos(self):
        """
        Caracteriza cada segmento encontrado
        """
        caracterizacion = self.features_df.groupBy("segmento").agg(
            count("customer_id").alias("num_clientes"),
            mean("age").alias("edad_promedio"),
            mean("gasto_total").alias("gasto_promedio"),
            mean("total_compras").alias("compras_promedio"),
            mean("probabilidad_compra").alias("propension_promedio"),
            mean("dias_desde_ultima_compra").alias("inactividad_promedio")
        ).orderBy("segmento")
        
        print("\nCaracterización de Segmentos:")
        caracterizacion.show()
        
        # Asignar nombres a segmentos basado en características
        segment_names = {
            0: "CLIENTES_LEALES",
            1: "CLIENTES_NUEVOS", 
            2: "CLIENTES_INACTIVOS",
            3: "CLIENTES_POTENCIALES"
        }
        
        self.features_df = self.features_df.withColumn(
            "segmento_nombre",
            when(col("segmento") == 0, segment_names[0])
            .when(col("segmento") == 1, segment_names[1])
            .when(col("segmento") == 2, segment_names[2])
            .when(col("segmento") == 3, segment_names[3])
        )
        
        return caracterizacion


In [28]:
# Segmentar clientes
segmentador = SegmentadorClientesSpark(n_segmentos=4)
features_df = segmentador.segmentar_clientes(features_df)



Segmentando clientes con K-Means...

Caracterización de Segmentos:
+--------+------------+-----------------+------------------+------------------+--------------------+--------------------+
|segmento|num_clientes|    edad_promedio|    gasto_promedio|  compras_promedio| propension_promedio|inactividad_promedio|
+--------+------------+-----------------+------------------+------------------+--------------------+--------------------+
|       0|      296263|34.52970840098156|33753.255109966674|25.826137587211363|  0.2934980515862676|   86.66386622696726|
|       1|       30889| 37.7190909385218|236834.32503809387|159.88669105506816|  0.6537675608665499|   34.34073618440222|
|       2|        1611|36.94909993792675|17527.707813864403|13.274363749224085| 0.09156572396028292|  353.17070142768466|
|       3|      146953| 41.9612393078059| 9123.882200546754|  6.42791232570958|0.002118582968920...|    477.738773621498|
+--------+------------+-----------------+------------------+------------------+

In [29]:
# Mostrar distribución de segmentos
print("\nDistribución de clientes por segmento:")
features_df.groupBy("segmento_nombre").agg(
    count("customer_id").alias("num_clientes"),
    (count("customer_id") / features_df.count() * 100).alias("porcentaje")
).show()


Distribución de clientes por segmento:
+--------------------+------------+-------------------+
|     segmento_nombre|num_clientes|         porcentaje|
+--------------------+------------+-------------------+
|  CLIENTES_INACTIVOS|        1611|0.33864742829755573|
|     CLIENTES_NUEVOS|       30889|  6.493159784409186|
|     CLIENTES_LEALES|      296263| 62.277283084865765|
|CLIENTES_POTENCIALES|      146953|   30.8909097024275|
+--------------------+------------+-------------------+



<a id='5'></a>
<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 5. Modelo Prescriptivo en PySpark

<div style="background-color:#9B9B9B; color:white; padding:10px; border-radius:5px; font-size:13px">

## 1. Contexto General
---

La clase **ModeloPrescriptivoSpark** implementa un sistema de recomendación comercial **prescriptivo** que combina segmentación de clientes, métricas de productos y reglas de negocio para generar acciones personalizadas por cliente o segmento.  
Está diseñado para ejecutarse de forma eficiente en entornos distribuidos mediante **PySpark**, con optimizaciones como *broadcast joins*, *caching* y transformaciones vectorizadas.

El modelo se alimenta de tres fuentes:

- **df_articles**: catálogo de artículos.
- **features_df**: clientes ya segmentados y enriquecidos con features (incluyendo probabilidad de compra).
- **df_ventas**: historial transaccional.

El resultado es un conjunto de estrategias comerciales accionables, listas para campañas de marketing, CRM o automatización.

## 2. Estructura General de la Clase
---

    class ModeloPrescriptivoSpark:
        ├── __init__                        # Inicializa fuentes y caché
        ├── calcular_metricas_articulos     # Métricas de rendimiento por artículo
        ├── generar_estrategias_segmento    # Define estrategias por segmento
        ├── generar_consolidado_optimizado  # Ensambla todas las recomendaciones
        ├── exportar_consolidado_parquet    # Guarda resultados en Parquet
        ├── recomendar_acciones_cliente     # Recomendación individual
        ├── generar_reporte_prescriptivo    # Resumen legible de estrategias
        ├── ejecutar_pipeline_completo      # Orquesta todo el flujo
        └── limpiar_cache                   # Libera memoria

## 3. Constructor: **__init__**
---

**Propósito:**

Inicializar las fuentes de datos y preparar el entorno para procesamiento distribuido.

    def __init__(self, df_articles, features_df, df_ventas):
        self.df_articles = df_articles
        self.features_df = features_df
        self.df_ventas = df_ventas
        self.recomendaciones_segmento = {}
        self.df_articles_metrics = None
        self.df_consolidado = None
        
        # Cachear DataFrames si son reutilizados
        self.df_articles.cache()
        self.features_df.cache()

- Se almacenan referencias a los DataFrames de entrada.
- Se activa el **caché** en los DataFrames más utilizados para evitar recomputación.

## 4. Método **calcular_metricas_articulos()**
---

**Objetivo:**

Calcular métricas clave de rendimiento por artículo, útiles para decisiones de inventario, pricing y promoción.

**Métricas calculadas por **article_id**:**

- **Total de ventas**:

  $\text{total\_ventas}_a = \text{count}(\text{customer\_id})$

- **Precio promedio**:

  $\text{precio\_promedio}_a = \frac{1}{n} \sum_{i=1}^{n} \text{price}_i$

- **Desviación estándar del precio**:

  $\text{std\_precio}_a = \sqrt{\frac{1}{n-1} \sum_{i=1}^{n} (\text{price}_i - \bar{p})^2}$

- **Clientes únicos**:

  $\text{clientes\_unicos}_a = \text{countDistinct}(\text{customer\_id})$

**Optimización:**

- Se usa **broadcast()** al hacer el *join* con **df_articles**, asumiendo que el catálogo es pequeño.
- Los valores nulos se rellenan con **0** para evitar sesgos en análisis posteriores.

**Resultado:**

DataFrame **df_articles_metrics** con métricas agregadas por artículo.

## 5. Método **generar_estrategias_segmento()**
---

**Objetivo:**

Definir una **estrategia comercial** por segmento de cliente, basada en su comportamiento promedio.

**Agregaciones por segmento:**

- Número de clientes
- Gasto promedio
- Frecuencia de compras
- Propensión promedio a comprar
- Inactividad promedio (días desde última compra)
- Edad promedio

**Lógica de asignación de estrategia:**

Se aplica una regla jerárquica mediante expresiones condicionales:

$\text{estrategia}_s = 
    \begin{cases}
        \text{PREMIUM} & \text{si } \text{gasto\_promedio} > 1000 \land \text{propension\_promedio} > 0.7 \\
        \text{CRECIMIENTO} & \text{si } \text{gasto\_promedio} > 500 \land \text{propension\_promedio} > 0.5 \\
        \text{REACTIVACIÓN} & \text{si } \text{inactividad\_promedio} > 90 \\
        \text{BASE} & \text{en otro caso}
    \end{cases}$

**Acciones asociadas a cada estrategia:**

- **PREMIUM**: productos exclusivos, descuentos altos, acceso anticipado.
- **CRECIMIENTO**: cross-selling, fidelización básica.
- **REACTIVACIÓN**: campañas agresivas de win-back.
- **BASE**: ofertas económicas y contenido inspiracional.

**Resultado:**

- **self.recomendaciones_segmento**: diccionario local con estrategias y acciones.
- **self.estrategias_df**: DataFrame listo para *join* con clientes.

## 6. Método **generar_consolidado_optimizado()**
---

**Objetivo:**

Construir un único DataFrame con **todas las variables prescriptivas** por cliente, en una sola pasada para maximizar eficiencia.

**Columnas generadas:**

1. **Canal recomendado**:

   $\text{canal\_recomendado} = 
       \begin{cases}
           \text{Email} & \text{si } \text{fashion\_news\_frequency} \neq \text{None} \\
           \text{Push Notification} & \text{en otro caso}
       \end{cases}$

2. **Frecuencia de contacto**:

   $\text{frecuencia\_contacto} = 
       \begin{cases}
           \text{Semanal} & \text{si } \text{probabilidad\_compra} > 0.7 \\
           \text{Quincenal} & \text{en otro caso}
       \end{cases}$

3. **Nivel de descuento**:

   $\text{nivel\_descuento} = 
       \begin{cases}
           \text{15-20\%} & \text{si } \text{gasto\_total} > 1000 \\
           \text{10-15\%} & \text{si } \text{gasto\_total} > 500 \\
           \text{5-10\%} & \text{en otro caso}
       \end{cases}$

**Optimizaciones:**

- Uso de **broadcast()** al unir con **estrategias_df**.
- Reparticionamiento por **segmento_nombre** para mejorar escritura paralela.

Resultado: **df_consolidado** listo para exportación o consulta.

## 7. Método **exportar_consolidado_parquet(output_path)**
---

**Objetivo:**

Persistir el DataFrame consolidado en formato **Parquet**, eficiente para lectura/escritura y compresión.

- Usa modo **overwrite**.
- Compresión por defecto: **Snappy** (rápida y eficaz).
- Crea directorio si no existe.
- Muestra estadísticas finales (total de registros).

## 8. Método **recomendar_acciones_cliente(customer_id)**
---

**Objetivo:**

Devolver un **dict** con recomendaciones personalizadas para un cliente específico.

**Campos devueltos:**

- **segmento**
- **probabilidad_compra**
- **gasto_total**
- **estrategia_recomendada**
- **acciones_prioritarias** (lista)
- **canal_recomendado**
- **frecuencia_contacto**
- **nivel_descuento**

**Optimización:**

- Si **df_consolidado** ya existe, se usa directamente (evita recálculo).
- Manejo de errores si el cliente no existe.

## 9. Método **generar_reporte_prescriptivo()**
---

**Objetivo:**

Imprimir un resumen legible en consola con las estrategias por segmento.

Incluye:

- Nombre del segmento
- Estrategia asignada
- Tamaño del segmento
- Métricas clave (gasto, propensión)
- Top 3 acciones recomendadas

## 10. Método **ejecutar_pipeline_completo(exportar=True)**
---

**Objetivo:**

Orquestar todo el flujo de forma secuencial y eficiente.

**Pasos ejecutados:**

1. Calcular métricas de artículos
2. Generar estrategias por segmento
3. Consolidar recomendaciones por cliente
4. Mostrar reporte prescriptivo
5. (Opcional) Exportar a Parquet

Devuelve el DataFrame consolidado final.

## 11. Método **limpiar_cache()**
---

**Objetivo:**

Liberar memoria al eliminar DataFrames previamente cacheados.

    self.df_articles.unpersist()
    self.features_df.unpersist()

Útil al finalizar el pipeline para evitar uso innecesario de memoria en clusters compartidos.

In [30]:
class ModeloPrescriptivoSpark:
    """
    Modelo prescriptivo optimizado para recomendar acciones comerciales basado en métricas de clientes y ventas.
    """

    def __init__(self, df_articles, features_df, df_ventas):
        self.df_articles = df_articles
        self.features_df = features_df
        self.df_ventas = df_ventas
        self.recomendaciones_segmento = {}
        self.df_articles_metrics = None
        self.df_consolidado = None
        
        # Cachear DataFrames si son reutilizados
        self.df_articles.cache()
        self.features_df.cache()

    # -------------------------------------------------------------------------
    def calcular_metricas_articulos(self):
        """
        Calcula métricas de rendimiento por artículo (ventas, precio promedio, dispersión, etc.)
        Optimizado con broadcast hint y reparticionamiento.
        """
        print("Calculando métricas de artículos...")

        ventas_articulo = (
            self.df_ventas
            .groupBy("article_id")
            .agg(
                count("customer_id").alias("total_ventas"),
                mean("price").alias("precio_promedio"),
                stddev("price").alias("std_precio"),
                countDistinct("customer_id").alias("clientes_unicos"),
            )
        )

        # Broadcast del DataFrame más pequeño para optimizar join
        self.df_articles_metrics = (
            self.df_articles
            .join(broadcast(ventas_articulo), "article_id", "left")
            .fillna(0, subset=["total_ventas", "precio_promedio", "std_precio", "clientes_unicos"])
        )

        return self.df_articles_metrics

    # -------------------------------------------------------------------------
    def generar_estrategias_segmento(self):
        """
        Genera estrategias prescriptivas por segmento de cliente.
        Optimizado: calcula estrategias una sola vez y las aplica via broadcast.
        """
        print("Generando estrategias prescriptivas por segmento...")

        # Agregar métricas de comportamiento por segmento (una sola pasada)
        segment_stats = (
            self.features_df
            .groupBy("segmento_nombre")
            .agg(
                count("customer_id").alias("num_clientes"),
                mean("gasto_total").alias("gasto_promedio"),
                mean("total_compras").alias("frecuencia_compras"),
                mean("probabilidad_compra").alias("propension_promedio"),
                mean("dias_desde_ultima_compra").alias("inactividad_promedio"),
                mean("age").alias("edad_promedio"),
            )
        )

        # Definir estrategias usando expresiones SQL de Spark (más eficiente)
        estrategias_df = segment_stats.withColumn(
            "estrategia",
            when((col("gasto_promedio") > 1000) & (col("propension_promedio") > 0.7), lit("PREMIUM"))
            .when((col("gasto_promedio") > 500) & (col("propension_promedio") > 0.5), lit("CRECIMIENTO"))
            .when(col("inactividad_promedio") > 90, lit("REACTIVACIÓN"))
            .otherwise(lit("BASE"))
        )

        # Mapeo de acciones por estrategia
        acciones_map = {
            "PREMIUM": [
                "Ofrecer productos exclusivos y ediciones limitadas",
                "Descuentos personalizados (15-20%)",
                "Acceso early a nuevos lanzamientos",
                "Programa de recompensas VIP",
                "Envío gratuito prioritario",
            ],
            "CRECIMIENTO": [
                "Recomendaciones cross-selling inteligentes",
                "Descuentos ocasionales (10-15%)",
                "Programa de fidelización básico",
                "Email marketing personalizado",
                "Cupones de bienvenida recurrentes",
            ],
            "REACTIVACIÓN": [
                "Campañas de win-back con descuentos agresivos (25-30%)",
                "Recordatorios de carrito abandonado",
                "Ofertas personalizadas basadas en historial",
                "Encuestas de satisfacción",
                "Programa de referidos",
            ],
            "BASE": [
                "Productos económicos destacados",
                "Descuentos por volumen",
                "Campañas estacionales",
                "Contenido educativo y de inspiración",
                "Pruebas gratuitas de servicios",
            ]
        }

        # Convertir a diccionario local para reporte
        estrategias = {}
        for row in estrategias_df.collect():
            segmento = row["segmento_nombre"]
            estrategia = row["estrategia"]
            
            estrategias[segmento] = {
                "estrategia": estrategia,
                "acciones": acciones_map.get(estrategia, []),
                "num_clientes": row["num_clientes"],
                "gasto_promedio": builtins.round(row["gasto_promedio"], 2),
                "propension_promedio": builtins.round(row["propension_promedio"], 3),
            }

        self.recomendaciones_segmento = estrategias
        
        # Guardar DataFrame con estrategias para join posterior
        self.estrategias_df = estrategias_df
        
        return estrategias

    # -------------------------------------------------------------------------
    def generar_consolidado_optimizado(self):
        """
        Genera el DataFrame consolidado con todas las columnas calculadas de forma vectorizada.
        Esto evita múltiples pasadas sobre los datos.
        """
        print("Generando DataFrame consolidado optimizado...")
        
        # Join con estrategias (broadcast para optimizar)
        df_con_estrategia = self.features_df.join(
            broadcast(self.estrategias_df.select("segmento_nombre", "estrategia")),
            "segmento_nombre",
            "left"
        )
        
        # Calcular todas las columnas adicionales en una sola transformación
        self.df_consolidado = df_con_estrategia.withColumn(
            "canal_recomendado",
            when(
                (col("fashion_news_frequency").isNotNull()) & 
                (col("fashion_news_frequency") != "None"),
                lit("Email")
            ).otherwise(lit("Push Notification"))
        ).withColumn(
            "frecuencia_contacto",
            when(col("probabilidad_compra") > 0.7, lit("Semanal"))
            .otherwise(lit("Quincenal"))
        ).withColumn(
            "nivel_descuento",
            when(col("gasto_total") > 1000, lit("15-20%"))
            .when(col("gasto_total") > 500, lit("10-15%"))
            .otherwise(lit("5-10%"))
        )
        
        # Reparticionar para optimizar escritura
        self.df_consolidado = self.df_consolidado.repartition(10, "segmento_nombre")
        
        return self.df_consolidado

    # -------------------------------------------------------------------------
    def exportar_consolidado_parquet(self, output_path=None):
        """
        Exporta el DataFrame consolidado en formato Parquet.
        """
        if output_path is None:
            output_path = GOLD
        if self.df_consolidado is None:
            print("Generando consolidado antes de exportar...")
            self.generar_consolidado_optimizado()

        # Crear directorio si no existe
        os.makedirs(output_path, exist_ok=True)
        
        full_path = f"{output_path}/consolidado_prescriptivo.parquet"
        
        print(f"Exportando consolidado a {full_path}...")
        
        # Escribir en Parquet con compresión snappy (por defecto y eficiente)
        self.df_consolidado.write.mode("overwrite").parquet(full_path)
        
        print(f"Consolidado exportado exitosamente a {full_path}")
        
        # Mostrar estadísticas
        total_registros = self.df_consolidado.count()
        print(f"Total de registros exportados: {total_registros:,}")
        
        return full_path

    # -------------------------------------------------------------------------
    def recomendar_acciones_cliente(self, customer_id):
        """
        Genera recomendaciones específicas para un cliente.
        Optimizado: usa el consolidado si está disponible.
        """
        # Si tenemos consolidado, usar ese (más rápido)
        if self.df_consolidado is not None:
            cliente_data = self.df_consolidado.filter(col("customer_id") == customer_id)
        else:
            cliente_data = self.features_df.filter(col("customer_id") == customer_id)

        if cliente_data.count() == 0:
            return {"error": f"Cliente {customer_id} no encontrado"}

        cliente = cliente_data.first()
        
        # Si usamos consolidado, las columnas ya están calculadas
        if self.df_consolidado is not None:
            recomendaciones = {
                "customer_id": customer_id,
                "segmento": cliente["segmento_nombre"],
                "probabilidad_compra": builtins.round(float(cliente["probabilidad_compra"]), 3),
                "gasto_total": builtins.round(float(cliente["gasto_total"]), 2),
                "estrategia_recomendada": cliente["estrategia"],
                "acciones_prioritarias": self.recomendaciones_segmento.get(
                    cliente["segmento_nombre"], {}
                ).get("acciones", []),
                "canal_recomendado": cliente["canal_recomendado"],
                "frecuencia_contacto": cliente["frecuencia_contacto"],
                "nivel_descuento": cliente["nivel_descuento"],
            }
        else:
            # Lógica original si no hay consolidado
            segmento = cliente["segmento_nombre"]
            estrategia_segmento = self.recomendaciones_segmento.get(segmento, {})

            canal = (
                "Email"
                if cliente["fashion_news_frequency"] and cliente["fashion_news_frequency"] != "None"
                else "Push Notification"
            )
            frecuencia = "Semanal" if cliente["probabilidad_compra"] > 0.7 else "Quincenal"

            gasto_total = cliente["gasto_total"]
            if gasto_total > 1000:
                descuento = "15-20%"
            elif gasto_total > 500:
                descuento = "10-15%"
            else:
                descuento = "5-10%"

            recomendaciones = {
                "customer_id": customer_id,
                "segmento": segmento,
                "probabilidad_compra": builtins.round(float(cliente["probabilidad_compra"]), 3),
                "gasto_total": builtins.round(float(gasto_total), 2),
                "estrategia_recomendada": estrategia_segmento.get("estrategia", "BASE"),
                "acciones_prioritarias": estrategia_segmento.get("acciones", []),
                "canal_recomendado": canal,
                "frecuencia_contacto": frecuencia,
                "nivel_descuento": descuento,
            }

        return recomendaciones

    # -------------------------------------------------------------------------
    def generar_reporte_prescriptivo(self):
        """
        Muestra un resumen legible de las estrategias prescriptivas por segmento.
        """
        print("\n" + "=" * 80)
        print("REPORTE PRESCRIPTIVO - ESTRATEGIAS POR SEGMENTO")
        

        for segmento, info in self.recomendaciones_segmento.items():
            print(f"\nSEGMENTO: {segmento}")
            print(f"Estrategia: {info['estrategia']}")
            print(f"Clientes: {info['num_clientes']:,}")
            print(f"Gasto promedio: ${info['gasto_promedio']:,.2f}")
            print(f"Propensión promedio: {info['propension_promedio']:.1%}")
            print("Acciones recomendadas:")
            for i, accion in enumerate(info["acciones"][:3], 1):
                print(f"  {i}. {accion}")

    # -------------------------------------------------------------------------
    def ejecutar_pipeline_completo(self, exportar=True):
        """
        Ejecuta todo el pipeline optimizado de forma secuencial.
        """
        print("Iniciando pipeline prescriptivo optimizado...")
        
        # 1. Calcular métricas de artículos
        self.calcular_metricas_articulos()
        
        # 2. Generar estrategias por segmento
        self.generar_estrategias_segmento()
        
        # 3. Generar consolidado optimizado
        self.generar_consolidado_optimizado()
        
        # 4. Generar reporte
        self.generar_reporte_prescriptivo()
        
        # 5. Exportar si se solicita
        if exportar:
            self.exportar_consolidado_parquet()
        
        print("\nPipeline completado exitosamente")
        
        return self.df_consolidado
    
    # -------------------------------------------------------------------------
    def limpiar_cache(self):
        """
        Libera memoria eliminando DataFrames cacheados.
        """
        self.df_articles.unpersist()
        self.features_df.unpersist()
        print("Cache liberado")

In [31]:
# Inicializar
modelo = ModeloPrescriptivoSpark(df_articles, features_df, df_transactions)


In [32]:
# Ejecutar pipeline completo (incluye exportación)
df_consolidado = modelo.ejecutar_pipeline_completo(exportar=True)


Iniciando pipeline prescriptivo optimizado...
Calculando métricas de artículos...
Generando estrategias prescriptivas por segmento...
Generando DataFrame consolidado optimizado...

REPORTE PRESCRIPTIVO - ESTRATEGIAS POR SEGMENTO

SEGMENTO: CLIENTES_INACTIVOS
Estrategia: REACTIVACIÓN
Clientes: 1,611
Gasto promedio: $17,527.71
Propensión promedio: 9.2%
Acciones recomendadas:
  1. Campañas de win-back con descuentos agresivos (25-30%)
  2. Recordatorios de carrito abandonado
  3. Ofertas personalizadas basadas en historial

SEGMENTO: CLIENTES_NUEVOS
Estrategia: CRECIMIENTO
Clientes: 30,889
Gasto promedio: $236,834.33
Propensión promedio: 65.4%
Acciones recomendadas:
  1. Recomendaciones cross-selling inteligentes
  2. Descuentos ocasionales (10-15%)
  3. Programa de fidelización básico

SEGMENTO: CLIENTES_LEALES
Estrategia: BASE
Clientes: 296,263
Gasto promedio: $33,753.26
Propensión promedio: 29.3%
Acciones recomendadas:
  1. Productos económicos destacados
  2. Descuentos por volumen
  

# 6. Dashboard y Visualización de Resultados

In [33]:
class DashboardPrescriptivoSpark:
    """
    Dashboard para visualizar resultados del modelo prescriptivo
    """
    
    def __init__(self, modelo_prescriptivo, features_df):
        self.modelo = modelo_prescriptivo
        self.features_df = features_df
        
    def mostrar_metricas_globales(self):
        """
        Muestra métricas globales del modelo
        """
        print("\n" + "="*60)
        print("DASHBOARD - MÉTRICAS GLOBALES")
        print("="*60)
        
        # Métricas básicas
        total_clientes = self.features_df.count()
        clientes_alta_propension = self.features_df.filter(col("probabilidad_compra") > 0.5).count()
        propension_promedio = self.features_df.agg(mean("probabilidad_compra")).collect()[0][0]
        
        print(f"TOTAL CLIENTES: {total_clientes:,}")
        print(f"Clientes con alta propensión (>50%): {clientes_alta_propension:,} ({clientes_alta_propension/total_clientes*100:.1f}%)")
        print(f"Propensión promedio: {propension_promedio:.2%}")
        
        # Distribución por segmento
        print(f"\nDISTRIBUCIÓN POR SEGMENTO:")
        dist_segmentos = self.features_df.groupBy("segmento_nombre").agg(
            count("customer_id").alias("clientes"),
            (count("customer_id") / total_clientes * 100).alias("porcentaje")
        ).collect()
        
        for row in dist_segmentos:
            print(f"   - {row['segmento_nombre']}: {row['clientes']:,} ({row['porcentaje']:.1f}%)")
    
    def mostrar_ejemplos_recomendaciones(self, n_ejemplos=5):
        """
        Muestra ejemplos de recomendaciones para clientes
        """
        print(f"\nEJEMPLOS DE RECOMENDACIONES PARA CLIENTES:")
        print("-" * 70)
        
        # Seleccionar clientes aleatorios de cada segmento
        for segmento in self.features_df.select("segmento_nombre").distinct().collect():
            segmento_nombre = segmento["segmento_nombre"]
            
            cliente_ejemplo = self.features_df.filter(col("segmento_nombre") == segmento_nombre) \
                .select("customer_id").limit(1).collect()
            
            if cliente_ejemplo:
                customer_id = cliente_ejemplo[0]["customer_id"]
                recomendacion = self.modelo.recomendar_acciones_cliente(customer_id)
                
                print(f"\nCLIENTE: {recomendacion['customer_id']}")
                print(f" Segmento: {recomendacion['segmento']}")
                print(f" Propensión: {recomendacion['probabilidad_compra']:.1%}")
                print(f" Gasto total: ${recomendacion['gasto_total']:,.2f}")
                print(f" Estrategia: {recomendacion['estrategia_recomendada']}")
                print(f" Canal: {recomendacion['canal_recomendado']}")
                print(f" Frecuencia: {recomendacion['frecuencia_contacto']}")
                print(f" Descuento: {recomendacion['nivel_descuento']}")
                print(f" Acciones:")
                for accion in recomendacion['acciones_prioritarias'][:2]:
                    print(f"  - {accion}")


In [34]:
# Ejecutar dashboard
dashboard = DashboardPrescriptivoSpark(modelo, features_df)
dashboard.mostrar_metricas_globales()
dashboard.mostrar_ejemplos_recomendaciones()



DASHBOARD - MÉTRICAS GLOBALES
TOTAL CLIENTES: 475,716
Clientes con alta propensión (>50%): 107,606 (22.6%)
Propensión promedio: 22.62%

DISTRIBUCIÓN POR SEGMENTO:
   - CLIENTES_INACTIVOS: 1,611 (0.3%)
   - CLIENTES_NUEVOS: 30,889 (6.5%)
   - CLIENTES_LEALES: 296,263 (62.3%)
   - CLIENTES_POTENCIALES: 146,953 (30.9%)

EJEMPLOS DE RECOMENDACIONES PARA CLIENTES:
----------------------------------------------------------------------

CLIENTE: a45a983e4090b4d85edd8dedccc28223b7f0a868bd3968f8e872b9a038a40b61
 Segmento: CLIENTES_INACTIVOS
 Propensión: 0.0%
 Gasto total: $38,093.22
 Estrategia: REACTIVACIÓN
 Canal: Email
 Frecuencia: Quincenal
 Descuento: 15-20%
 Acciones:
  - Campañas de win-back con descuentos agresivos (25-30%)
  - Recordatorios de carrito abandonado

CLIENTE: 521ff2316426a6e1761c578ae4a0befac693ce7b7f6bde561ccf7e0ed5101ab6
 Segmento: CLIENTES_NUEVOS
 Propensión: 99.4%
 Gasto total: $313,240.68
 Estrategia: CRECIMIENTO
 Canal: Email
 Frecuencia: Semanal
 Descuento: 15-20%


In [35]:
spark.stop()